# Bronze Layer - Olist E-Commerce Data Ingestion

This notebook ingests raw Olist e-commerce CSV files from a Unity Catalog Volume into Delta Lake Bronze tables.

## Bronze Layer Responsibilities

- Preserve source data with minimal transformation
- Create Delta tables for each source dataset
- Add ingestion metadata for traceability
- Provide a reliable source for downstream Silver transformations

In [0]:
from pyspark.sql import functions as F

CATALOG = "ecommerce_lakehouse"
SCHEMA = "bronze"
RAW_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

print(f"Source: {RAW_VOLUME}")
print(f"Target: {CATALOG}.{SCHEMA}")

Source: /Volumes/ecommerce_lakehouse/bronze/raw
Target: ecommerce_lakehouse.bronze


In [0]:
files = dbutils.fs.ls(RAW_VOLUME)

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [0]:
datasets = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv"
}

In [0]:
def ingest_to_bronze(table_name, file_name):
    source_path = f"{RAW_VOLUME}/{file_name}"
    target_table = f"{CATALOG}.{SCHEMA}.{table_name}"

    print(f"Processing: {file_name}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(source_path)
    )

    bronze_df = (
        df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(file_name))
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(
        f"Created {target_table} "
        f"({bronze_df.count():,} rows, {len(bronze_df.columns)} columns)"
    )

In [0]:
ingest_to_bronze(
    "customers",
    "olist_customers_dataset.csv"
)

Processing: olist_customers_dataset.csv
Created ecommerce_lakehouse.bronze.customers (99,441 rows, 7 columns)


In [0]:
display(
    spark.table("ecommerce_lakehouse.bronze.customers")
    .limit(10)
)

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_ingested_at,_source_file
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,2026-09-20T12:55:14.624Z,olist_customers_dataset.csv


In [0]:
spark.sql("""
DESCRIBE DETAIL ecommerce_lakehouse.bronze.customers
""").display()

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a013eaac-fe81-4e4d-a5a5-9859011dfe72,ecommerce_lakehouse.bronze.customers,null,,2026-09-20T12:55:13.347Z,2026-09-20T12:55:19.000Z,List(),List(),1,3732309,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> aa4b2b7b-78ba-4fc9-a91d-5a128af57fad)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
for table_name, file_name in datasets.items():
    ingest_to_bronze(table_name, file_name)

Processing: olist_customers_dataset.csv
Created ecommerce_lakehouse.bronze.customers (99,441 rows, 7 columns)
Processing: olist_geolocation_dataset.csv
Created ecommerce_lakehouse.bronze.geolocation (1,000,163 rows, 7 columns)
Processing: olist_order_items_dataset.csv
Created ecommerce_lakehouse.bronze.order_items (112,650 rows, 9 columns)
Processing: olist_order_payments_dataset.csv
Created ecommerce_lakehouse.bronze.order_payments (103,886 rows, 7 columns)
Processing: olist_order_reviews_dataset.csv
Created ecommerce_lakehouse.bronze.order_reviews (99,224 rows, 9 columns)
Processing: olist_orders_dataset.csv
Created ecommerce_lakehouse.bronze.orders (99,441 rows, 10 columns)
Processing: olist_products_dataset.csv
Created ecommerce_lakehouse.bronze.products (32,951 rows, 11 columns)
Processing: olist_sellers_dataset.csv
Created ecommerce_lakehouse.bronze.sellers (3,095 rows, 6 columns)
Processing: product_category_name_translation.csv
Created ecommerce_lakehouse.bronze.product_categor

In [0]:
spark.sql("""
SHOW TABLES IN ecommerce_lakehouse.bronze
""").display()

database,tableName,isTemporary
bronze,customers,false
bronze,geolocation,false
bronze,order_items,false
bronze,order_payments,false
bronze,order_reviews,false
bronze,orders,false
bronze,product_category_translation,false
bronze,products,false
bronze,sellers,false


In [0]:
for table_name in datasets.keys():
    count = spark.table(
        f"{CATALOG}.{SCHEMA}.{table_name}"
    ).count()

    print(f"{table_name:<30} {count:>10,} rows")

customers                          99,441 rows
geolocation                     1,000,163 rows
order_items                       112,650 rows
order_payments                    103,886 rows
order_reviews                      99,224 rows
orders                             99,441 rows
products                           32,951 rows
sellers                             3,095 rows
product_category_translation           71 rows
